In [1]:
# Import some libraries and especially load environment variables
from langchain_openai  import AzureChatOpenAI
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv()) # read local .env file
# set an environment varibale
os.environ["LANGCHAIN_PROJECT"] = "01_basic"

print(os.getenv("AZURE_ENDPOINT"))
print(os.getenv("OPENAI_API_BASE"))


https://alkopenai2.openai.azure.com/
https://alkopenai2.openai.azure.com/


In [2]:
llm = AzureChatOpenAI(
	temperature=0,
    openai_api_version="2023-09-01-preview",
    deployment_name="GPT-4o-mini", #Deployment name
    azure_endpoint=os.environ["OPENAI_API_BASE"],
    model_name="GPT-4o-mini"
)

In [3]:
# I the import my custom function to work with audio and videp
from plugins.AudioVideo import AudioVideo
from plugins.Summarization import Summarization

summarization = Summarization()
audio_video = AudioVideo()

# # you can test the function
# audio_video.extract_audio("C:\\temp\\300.mp4")
# audio_video.extract_audio("/Users/gianmariaricci/develop/montaggi/RDP.mp4");

In [4]:
# Then create tools list
from langchain.tools import Tool

tools = [
    Tool.from_function(
        func=audio_video.extract_audio,
        name="ExtractAudio",
        description="extract audio in wav format from an mp4 file",
    ),
    Tool.from_function(
        func=audio_video.transcript_timeline,
        name="TranscriptTimeline",
        description="Transcript audio from a wav file to a full transcription",
    ),
    Tool.from_function(
        func=summarization.summarize_timeline,
        name="SummarizeTimeline",
        description="Take a full transcription and create a summarized timeline.",
    ),
]

In [5]:
from langchain import hub

# Get the prompt to use - you can modify this!
prompt = hub.pull("hwchase17/openai-functions-agent")
prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [6]:
# Finally you can create the agent
from langchain.agents import create_tool_calling_agent
agent = create_tool_calling_agent(tools=tools, llm=llm, prompt=prompt)

In [7]:
# Everthing is ready to start the conversation
from langchain.agents import AgentExecutor

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
agent_executor.invoke({"input": "I need a summarized timeline from video /Users/gianmariaricci/develop/montaggi/RDP.mp4"})



> Entering new AgentExecutor chain...

Invoking: `ExtractAudio` with `/Users/gianmariaricci/develop/montaggi/RDP.mp4`


Extracting audio file from video /Users/gianmariaricci/develop/montaggi/RDP.mp4
/Users/gianmariaricci/develop/montaggi/RDP.wav
Invoking: `TranscriptTimeline` with `/Users/gianmariaricci/develop/montaggi/RDP.wav`


Extracting transcript from audio file /Users/gianmariaricci/develop/montaggi/RDP.wav


/Users/gianmariaricci/develop/github/ai-playground/src/python/various/various/lib/python3.12/site-packages/whisper/__init__.py:150: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental featur

extracted 50 audio segments
00:00	 Hi, welcome to this new video about UBKey,
00:03	 where we will deal with Windows remote desktop
00:07	 and how we can use UBKey with the Windows remote desktop protocol.
00:13	 This question is one of the I received the most.
00:18	 Luckily, we have a simple answer.
00:21	 Remote desktop protocol in Windows
00:23	 notively support UBKey.
00:25	 So for this demo,
00:26	 I'm going to use a remote Windows machine in Azure.
00:29	 It is in the other part of the global.
00:32	 We are in Europe, the machine is seen in United States.
00:34	 So I'll show you that you only need
00:36	 a remote desktop to get started and use your UBKey
00:40	 inside your remote Windows machine.
00:43	 The option is enabled by default,
00:45	 but if you want to be sure that it is indeed enabled,
00:49	 you need to go to local resources and go to more,
00:54	 and be sure that you have selected this Markart or Windows
00:57	 Hello for Business and Webout and Windows Elor Security

{'input': 'I need a summarized timeline from video /Users/gianmariaricci/develop/montaggi/RDP.mp4',
 'output': 'Here is the summarized timeline from the video:\n\n- **00:00** - Introduction to UBKey and Remote Desktop\n- **00:21** - Enabling UBKey for Remote Desktop\n- **01:17** - Signing In with UBKey on Remote Machine\n- **02:46** - Benefits of Using UBKey with Remote Desktop\n- **03:05** - Conclusion and Summary\n\nIf you need any further details or assistance, feel free to ask!'}